> 在线讲义：[第 2 章 · Tensor 内部：一块内存 + 一张"怎么读"的说明书](https://wikieden.github.io/pytorch-inside-out/lessons/02-tensor-stride.html) · 本书《PyTorch 从里到外》

# 第 2 章 · Tensor 内部：一块内存 + 一张"怎么读"的说明书

本章回答：一个 Tensor 里面到底存了什么？为什么 `x.t()`、`x[:, 1:]` 几乎不花时间？什么叫 contiguous，为什么 `view` 有时会报错？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/02-tensor-stride.md`（讲解、动画、练习都在讲义里）。对应原文：00-overview.md §2.1；官方文档 Tensor Views（〔在线〕docs.pytorch.org/docs/stable/tensor_view.html）；ezyang「PyTorch internals」Tensor 一节（〔在线〕）

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 说明书 | pos = offset + Σ iₖ sₖ | `describe(x)`、`position(0, (3, 1), (1, 2))` | stride (3, 1)；位置 5 |
| 第二步 转置 | stride 对调 | `describe(x.t())` | (3, 2)、stride (1, 3)、不连续 |
| 第二步 切片 | offset 变 | `describe(x[:, 1:])` | offset 1 |
| 第三步 view 报错 | 按行读 ≠ storage 顺序 | `try_view_flat(x.t())` | 报错；reshape 得 [0, 3, 1, 4, 2, 5] |
| 第四步 stride 0 | expand 不复制 | `b.expand(4, 3).stride()` | (0, 1)；repeat 是 (3, 1) |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `describe`**（来自 `ch02_tensor_stride.py`）：说明书：shape、stride、offset、是否连续。

In [ ]:
def describe(t: torch.Tensor) -> dict:
    """说明书：shape、stride、offset、是否连续。"""
    return dict(shape=tuple(t.shape), stride=t.stride(), offset=t.storage_offset(), contiguous=t.is_contiguous())

**函数 `position`**（来自 `ch02_tensor_stride.py`）：元素在 storage 里的位置 = offset + Σ i_k × stride_k。

In [ ]:
def position(offset: int, stride: tuple, index: tuple) -> int:
    """元素在 storage 里的位置 = offset + Σ i_k × stride_k。"""
    return offset + sum(i * s for i, s in zip(index, stride))

**函数 `try_view_flat`**（来自 `ch02_tensor_stride.py`）：不连续时 view 会报错；返回 (是否成功, 结果或报错开头)。

In [ ]:
def try_view_flat(t: torch.Tensor):
    """不连续时 view 会报错；返回 (是否成功, 结果或报错开头)。"""
    try:
        return True, t.view(-1).tolist()
    except RuntimeError as e:
        return False, str(e).split(".")[0]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
x = torch.arange(6).view(2, 3)

In [ ]:
print("\n第一步 一排数 + 说明书")
print("x 的说明书：", describe(x))
print("x[1, 2] 的位置 = 0 + 1×3 + 2×1 =", position(0, x.stride(), (1, 2)), "→ 值", x[1, 2].item())

In [ ]:
print("\n第二步 view 只改说明书")
t = x.t()
print("x.t() 的说明书：", describe(t))
print("x.t()[2, 1] 位置 =", position(0, t.stride(), (2, 1)))
s = x[:, 1:]
print("x[:, 1:] 的说明书：", describe(s))
print("共用内存：", x.data_ptr() == t.data_ptr())

In [ ]:
print("\n第三步 contiguous")
print("x.t().view(6)：", try_view_flat(t))
print("x.t().contiguous().view(6)：", t.contiguous().view(6).tolist())
print("x.t().reshape(6)：", t.reshape(6).tolist())

In [ ]:
print("\n第四步 stride = 0")
b = torch.tensor([10, 20, 30])
print("b.expand(4, 3) 的 stride：", b.expand(4, 3).stride(), "；repeat 的 stride：", b.repeat(4, 1).stride())

## 3. 代码里才看得到的

1. **`x[:, 1:]` 也不连续**：shape (2, 2) 的标准 stride 是 (2, 1)，它是 (3, 1)，`is_contiguous()` 为 False。
2. **报错信息很长**：`view size is not compatible with input tensor's size and stride`，看到它就想到第三步。

## 4. 自己改一改

- 对 `torch.arange(24).view(2, 3, 4)` 调 `describe`，再 `permute(2, 0, 1)` 看 stride 怎么跟着换。
- 用 `position` 手算一个三维张量元素的位置，再用下标取出来对一对。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 一排数加说明书：shape 说几行几列，stride 说一步跳几格，offset 说从哪开始；
> view 只换说明书，转置跳着读不连续；stride 为 0 一份当多份。